In [ ]:
!pip install einops datasets --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import math

**Self Attention**

$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class SelfAttention(nn.Module):
  def __init__(self, d_model):
    super(SelfAttention, self).__init__()
    self.d_model = d_model #d_model = 512
    self.wq = nn.Linear(d_model, d_model)
    self.wk = nn.Linear(d_model, d_model)
    self.wv = nn.Linear(d_model, d_model)


  def forward(self, x):
    """
        Input:
            x -> (Batch Size, Sequence Length, Embedding Dimension)-> (2, 5, 256)
    """
    Q = self.wq(x) # (Batch Size, Sequence Length, Embedding Dimension)
    K = self.wk(x) # (Batch Size, Sequence Length, Embedding Dimension)
    V = self.wv(x) # (Batch Size, Sequence Length, Embedding Dimension)

    scores = Q@K.transpose(-2, -1) # (Batch Size, Sequence Length, Sequence Length) , outputs each tokens score against each token,
    scores /= math.sqrt(Q.size(-1))

    weights = torch.softmax(scores, dim=-1) # (Batch Size, Sequence Length, Sequence Length) , coomputes softmax across each single row

    return weights@V # (Batch Size, Sequence Length, Embedding Dimension)

In [ ]:
self_attention = SelfAttention(512)
x = torch.randn((2,5, 512))
attention = self_attention(x)
print(attention.shape)
print(attention)

torch.Size([2, 5, 512])
tensor([[[ 0.1166, -0.1181,  0.0454,  ..., -0.3491, -0.1585,  0.0060],
         [ 0.1601, -0.2749, -0.0186,  ..., -0.2177,  0.1456, -0.3369],
         [ 0.1547, -0.1925,  0.0153,  ..., -0.2903, -0.0635, -0.1498],
         [ 0.2399, -0.2508, -0.0172,  ..., -0.2807, -0.2782, -0.1910],
         [ 0.1265, -0.2447,  0.0302,  ..., -0.2300,  0.1940, -0.1759]],

        [[ 0.3098, -0.3575, -0.1949,  ..., -0.0138, -0.0758,  0.1222],
         [ 0.3849, -0.2860, -0.1645,  ..., -0.3474, -0.1659,  0.2782],
         [ 0.2896, -0.4330, -0.0417,  ...,  0.0325, -0.1638,  0.2373],
         [ 0.2848, -0.3640, -0.0361,  ..., -0.0785, -0.1892,  0.1832],
         [ 0.3764, -0.3953, -0.1059,  ..., -0.2084, -0.1729,  0.3882]]],
       grad_fn=<UnsafeViewBackward0>)


**Multihead Attention**


$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$

$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, heads):
    super(MultiHeadAttention, self).__init__()
    self.heads = heads
    self.embed_size = d_model
    self.head_dim = d_model // heads

    assert (
        self.head_dim * heads == d_model
    ), "Embedding size needs to be divisible by heads"

    self.wq = nn.Linear(d_model, d_model)
    self.wk = nn.Linear(d_model, d_model)
    self.wv = nn.Linear(d_model, d_model)

    self.wo = nn.Linear(d_model, d_model)

  def forward(self, x):
    B,S,D = x.shape

    Q = self.wq(x) # (Batch Size, Sequence Length, Embedding Dimension) -> (2, 5, 512)
    K = self.wk(x) # (Batch Size, Sequence Length, Embedding Dimension)
    V = self.wv(x) # (Batch Size, Sequence Length, Embedding Dimension)

    Q = Q.view(B, S, self.heads, self.head_dim).transpose(1,2) # (2, 8, 5, 64)
    K = K.view(B, S, self.heads, self.head_dim).transpose(1,2) # (2, 8, 5, 64)
    V = V.view(B, S, self.heads, self.head_dim).transpose(1,2) # (2, 8, 5, 64)

    scores = Q@K.transpose(-2, -1)
    scores /= math.sqrt(Q.size(-1)) # (2, 8, 5, 5) , for each head we check the attention score of one against all

    weights = torch.softmax(scores, dim=-1) # (2, 8, 5, 5)
    attention = weights@V # (2, 8, 5, 64)
    attention = attention.transpose(1,2).contiguous().view(B, S, D) # (2, 5, 512)
    return self.wo(attention)


In [ ]:
multi_attention = MultiHeadAttention(512, 8)
x = torch.randn((2,5, 512))
attention = multi_attention(x)
print(attention.shape)
print(attention)

torch.Size([2, 5, 512])
tensor([[[-0.1366,  0.2143, -0.0596,  ..., -0.2811,  0.0315, -0.0458],
         [-0.1317,  0.1289, -0.1272,  ..., -0.2445,  0.0258, -0.0324],
         [-0.0990,  0.2070, -0.1716,  ..., -0.3268,  0.0240, -0.0515],
         [-0.0265,  0.1602, -0.1754,  ..., -0.2300,  0.0029, -0.0995],
         [-0.1182,  0.2787, -0.1174,  ..., -0.2753, -0.0126, -0.0673]],

        [[-0.2244, -0.0706,  0.0604,  ...,  0.2737,  0.3723, -0.2947],
         [-0.1959,  0.0117, -0.0705,  ...,  0.1611,  0.3404, -0.3184],
         [-0.1444, -0.0169,  0.0039,  ...,  0.2275,  0.3787, -0.2107],
         [-0.2445, -0.1118,  0.0270,  ...,  0.2826,  0.3311, -0.2787],
         [-0.2418, -0.0549,  0.0129,  ...,  0.1817,  0.3177, -0.2921]]],
       grad_fn=<ViewBackward0>)


# Grouped Query Attention (GQA)

Grouped Query Attention (GQA) interpolates between Multi-Head Attention (MHA) and Multi-Query Attention (MQA) by grouping query heads to share a single key/value head.

---

## Mathematical Formulation

### 1. Head Mapping Function

For any query head $i \in \{1, \dots, H_Q\}$, the corresponding Key/Value head index $g(i)$ is defined as:

$$g(i) = \left\lfloor \frac{i - 1}{G} \right\rfloor + 1 \quad \text{where} \quad G = \frac{H_Q}{H_{KV}}$$

Here, $G$ represents the group size (number of query heads sharing a single key/value head).

---

### 2. Scaled Dot-Product Attention (Per Head)

For query head $i$, attention is computed using its projected Query matrix $Q_i$ alongside its assigned Key matrix $K_{g(i)}$ and Value matrix $V_{g(i)}$:

$$\text{Head}_i = \text{softmax}\left( \frac{Q_i K_{g(i)}^\top}{\sqrt{d_k}} \right) V_{g(i)}$$

---

### 3. Full Output Projection

All $H_Q$ attention head outputs are concatenated and projected back to the hidden dimension $d_{\text{model}}$:

$$\text{GQA}(X) = \text{Concat}\left(\text{Head}_1, \text{Head}_2, \dots, \text{Head}_{H_Q}\right) W^O$$

Where the linear projections are given by:

$$
\begin{aligned}
Q_i &= X W_i^Q \quad \text{for } i \in \{1, \dots, H_Q\} \\
K_j &= X W_j^K \quad \text{for } j \in \{1, \dots, H_{KV}\} \\
V_j &= X W_j^V \quad \text{for } j \in \{1, \dots, H_{KV}\}
\end{aligned}
$$

In [ ]:
class MultiHeadGroupedQueryAttention(nn.Module):
    def __init__(self, d_model, q_heads, kv_heads):
        super().__init__()

        assert d_model % q_heads == 0
        assert q_heads % kv_heads == 0

        self.d_model = d_model
        self.q_heads = q_heads
        self.kv_heads = kv_heads
        self.q_per_kv = q_heads // kv_heads
        self.head_dim = d_model // q_heads
        self.kv_dim = self.kv_heads * self.head_dim

        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, self.kv_dim, bias=False)
        self.wv = nn.Linear(d_model, self.kv_dim, bias=False)

        self.wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):

        B, S, D = x.shape

        Q = self.wq(x)
        K = self.wk(x)
        V = self.wv(x)

        q = rearrange(Q, "b s (h d) -> b h s d", h=self.q_heads, d=self.head_dim)
        k = rearrange(K, "b s (h d) -> b h s d", h=self.kv_heads, d=self.head_dim)
        v = rearrange(V, "b s (h d) -> b h s d", h=self.kv_heads, d=self.head_dim)

        # Repeat K/V for every query head in the group
        if self.q_per_kv > 1:

            k = rearrange(k, "b h s d -> b h 1 s d")
            k = k.expand(-1, -1, self.q_per_kv, -1, -1)
            k = rearrange(k, "b h g s d -> b (h g) s d")

            v = rearrange(v, "b h s d -> b h 1 s d")
            v = v.expand(-1, -1, self.q_per_kv, -1, -1)
            v = rearrange(v, "b h g s d -> b (h g) s d")

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        weights = torch.softmax(scores, dim=-1)

        attention = weights @ v

        out = rearrange(attention, "b h s d -> b s (h d)")
        out = self.wo(out)

        return out

In [ ]:
gq_attention = MultiHeadGroupedQueryAttention(512, 8, 2)
x = torch.randn((2,5, 512))
attention = gq_attention(x)
print(attention.shape)
print(attention)

torch.Size([2, 5, 512])
tensor([[[-0.0563, -0.0119, -0.1061,  ...,  0.1996, -0.0835, -0.2283],
         [-0.0685,  0.0776, -0.0230,  ...,  0.2106, -0.0632, -0.1414],
         [-0.0746, -0.0264, -0.0459,  ...,  0.1624, -0.0971, -0.2247],
         [-0.1305, -0.0017, -0.0881,  ...,  0.1125, -0.1201, -0.1479],
         [-0.0728,  0.0118, -0.0572,  ...,  0.1548, -0.0815, -0.1976]],

        [[-0.0802, -0.2284,  0.0271,  ...,  0.0399,  0.1995,  0.1672],
         [-0.0401, -0.2509,  0.0391,  ..., -0.0109,  0.3255,  0.2283],
         [ 0.0361, -0.2675,  0.0438,  ...,  0.0371,  0.2290,  0.1959],
         [-0.1046, -0.2093,  0.0174,  ...,  0.0730,  0.2093,  0.1734],
         [-0.0177, -0.2148,  0.0241,  ..., -0.0036,  0.2247,  0.2343]]],
       grad_fn=<UnsafeViewBackward0>)
